In [1]:
from pyexpat import features

import torch
import torch.nn as nn
from sympy.abc import epsilon


class Model(nn.Module):
    def __init__(self,num_features):
        super(Model, self).__init__()
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid= nn.Sigmoid()

    def forward(self,features):
        output = self.linear(features)
        output = self.sigmoid(output)
        return output

In [2]:
features = torch.randn(10,5)
model = Model(features.shape[1])
model(features)


tensor([[0.3864],
        [0.4184],
        [0.3230],
        [0.2262],
        [0.4529],
        [0.4762],
        [0.4842],
        [0.3723],
        [0.3619],
        [0.6739]], grad_fn=<SigmoidBackward0>)

In [3]:
model.linear.weight

Parameter containing:
tensor([[-0.1778,  0.3606,  0.3390,  0.1576, -0.2092]], requires_grad=True)

In [4]:
model.linear.bias

Parameter containing:
tensor([-0.2173], requires_grad=True)

In [5]:
from torchinfo import summary
summary(model,input_size=(10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Linear: 1-1                            [10, 1]                   6
├─Sigmoid: 1-2                           [10, 1]                   --
Total params: 6
Trainable params: 6
Non-trainable params: 0
Total mult-adds (M): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [8]:
class Model1(nn.Module):
    def __init__(self,num_features):
        super(Model1,self).__init__()
        self.linear1 = nn.Linear(num_features, 3)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(3, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self,features):
        output = self.linear1(features)
        output = self.relu(output)
        output = self.linear2(output)
        output = self.sigmoid(output)
        return output



In [9]:
features = torch.randn(10,5)
model = Model1(features.shape[1])
model(features)

tensor([[0.4588],
        [0.4865],
        [0.5007],
        [0.4404],
        [0.4133],
        [0.4333],
        [0.5401],
        [0.4778],
        [0.4942],
        [0.4720]], grad_fn=<SigmoidBackward0>)

In [11]:
summary(model,input_size=(10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model1                                   [10, 1]                   --
├─Linear: 1-1                            [10, 3]                   18
├─ReLU: 1-2                              [10, 3]                   --
├─Linear: 1-3                            [10, 1]                   4
├─Sigmoid: 1-4                           [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (M): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [12]:
class Model3(nn.Module):
    def __init__(self,num_features):
        super(Model3,self).__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, 3),
            nn.ReLU(),
            nn.Linear(3,10),
            nn.ReLU(),
            nn.Linear(10,1),
            nn.Sigmoid(),
        )
    def forward(self,fet):
        output = self.network(fet)
        return output

In [13]:
model3 = Model3(features.shape[1])
model3(features)
summary(model3,input_size=(10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model3                                   [10, 1]                   --
├─Sequential: 1-1                        [10, 1]                   --
│    └─Linear: 2-1                       [10, 3]                   18
│    └─ReLU: 2-2                         [10, 3]                   --
│    └─Linear: 2-3                       [10, 10]                  40
│    └─ReLU: 2-4                         [10, 10]                  --
│    └─Linear: 2-5                       [10, 1]                   11
│    └─Sigmoid: 2-6                      [10, 1]                   --
Total params: 69
Trainable params: 69
Non-trainable params: 0
Total mult-adds (M): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [16]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [17]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [18]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [19]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [20]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [21]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [22]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)
X_train_tensor.shape , y_train_tensor.shape

(torch.Size([455, 30]), torch.Size([455]))

In [27]:
class PredictionModel(nn.Module):
    def __init__(self,num_features):
        super(PredictionModel,self).__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, 8),
            nn.ReLU(),
            nn.Linear(8,4),
            nn.ReLU(),
            nn.Linear(4,1),
            nn.Sigmoid(),
        )
    def forward(self, fe):
        output = self.network(fe)
        return output


In [32]:
learning_rate = 0.5
epochs = 50

In [40]:

model4 = PredictionModel(X_train_tensor.shape[1])
optimizer = torch.optim.SGD(model4.parameters(), lr=learning_rate)
loss_function = nn.BCELoss()
for epoch in range(epochs):
    y_pred = model4(X_train_tensor.float())
    loss = loss_function(y_pred, y_train_tensor.reshape(-1,1).float())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


    print("epoch : ",epoch+1 , " loss : ",loss.item())

epoch :  1  loss :  0.6862908601760864
epoch :  2  loss :  0.6683571338653564
epoch :  3  loss :  0.6506124138832092
epoch :  4  loss :  0.6277137994766235
epoch :  5  loss :  0.5982691049575806
epoch :  6  loss :  0.5591662526130676
epoch :  7  loss :  0.5089028477668762
epoch :  8  loss :  0.45111849904060364
epoch :  9  loss :  0.3962189853191376
epoch :  10  loss :  0.3476217985153198
epoch :  11  loss :  0.30438098311424255
epoch :  12  loss :  0.26519691944122314
epoch :  13  loss :  0.23013149201869965
epoch :  14  loss :  0.1998380720615387
epoch :  15  loss :  0.1746392697095871
epoch :  16  loss :  0.15431740880012512
epoch :  17  loss :  0.13790754973888397
epoch :  18  loss :  0.12454622983932495
epoch :  19  loss :  0.11388573795557022
epoch :  20  loss :  0.10528473556041718
epoch :  21  loss :  0.09814205020666122
epoch :  22  loss :  0.09222117066383362
epoch :  23  loss :  0.08730802685022354
epoch :  24  loss :  0.08313056081533432
epoch :  25  loss :  0.0795439407229

In [51]:
# model evaluation
with torch.no_grad():
  y_pred = model4.forward(X_test_tensor.float())
  y_pred = (y_pred > 0.7).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')


Accuracy: 0.5220067501068115


# Batch training of data Manually
 ### Problems
- no standard interface for data
- no easy way to manage data transformation
- code looks complex
- shuffling and sampling manually is tough
- Batch management and parallelization requires huge effort

In [52]:
batch_size =32
model4 = PredictionModel(X_train_tensor.shape[1])
optimizer = torch.optim.SGD(model4.parameters(), lr=learning_rate)
loss_function = nn.BCELoss()
for epoch in range(epochs):
    for start in range(0, X_train_tensor.shape[0], batch_size):
        end = start + batch_size
        x_batch = X_train_tensor[start:end]
        y_batch = y_train_tensor[start:end]
        y_pred = model4(X_train_tensor.float())
        loss = loss_function(y_pred, y_train_tensor.reshape(-1,1).float())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


    print("epoch : ",epoch+1 , " loss : ",loss.item())

epoch :  1  loss :  0.1892617791891098
epoch :  2  loss :  0.06621932238340378
epoch :  3  loss :  0.0481848381459713
epoch :  4  loss :  0.04070138931274414
epoch :  5  loss :  0.036179497838020325
epoch :  6  loss :  0.03271821513772011
epoch :  7  loss :  0.029399806633591652
epoch :  8  loss :  0.026546167209744453
epoch :  9  loss :  0.02415715903043747
epoch :  10  loss :  0.022033629938960075
epoch :  11  loss :  0.020156368613243103
epoch :  12  loss :  0.018459338694810867
epoch :  13  loss :  0.01685461588203907
epoch :  14  loss :  0.015393143519759178
epoch :  15  loss :  0.01406643632799387
epoch :  16  loss :  0.012868143618106842
epoch :  17  loss :  0.011807643808424473
epoch :  18  loss :  0.010886035859584808
epoch :  19  loss :  0.010059413500130177
epoch :  20  loss :  0.00928331445902586
epoch :  21  loss :  0.00859951600432396
epoch :  22  loss :  0.007961085997521877
epoch :  23  loss :  0.007396900560706854
epoch :  24  loss :  0.006887699477374554
epoch :  25  

# Automated Approach

In [53]:
from sklearn.datasets import make_classification
import torch

In [54]:
X,y = make_classification(
    n_samples=10,
    n_features=2,
    n_classes=2,
    n_informative=2,
    n_redundant=0,
    random_state=42,
)

In [55]:
X

array([[ 1.06833894, -0.97007347],
       [-1.14021544, -0.83879234],
       [-2.8953973 ,  1.97686236],
       [-0.72063436, -0.96059253],
       [-1.96287438, -0.99225135],
       [-0.9382051 , -0.54304815],
       [ 1.72725924, -1.18582677],
       [ 1.77736657,  1.51157598],
       [ 1.89969252,  0.83444483],
       [-0.58723065, -1.97171753]])

In [56]:
X=torch.from_numpy(X).float()
y=torch.from_numpy(y).float()

In [63]:
from torch.utils.data import Dataset, DataLoader
class CustomDataset(Dataset):
    def __init__(self, fe, labels):
        self.features = fe
        self.labels = labels

    def __len__(self):
        return self.features.shape[0]

    def __getitem__(self, idx):
        # Here we can apply transformation to data like (resize/black&white/augmentation) image data OR (lowercasig/stop_words/lemmatization/stemming etc) in textual data
        return self.features[idx], self.labels[idx]

dataset = CustomDataset(X, y)

In [64]:
len(dataset)

10

In [65]:
 dataset[0]

(tensor([ 1.0683, -0.9701]), tensor(1.))

In [68]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

for batch_features, batch_labels in dataloader:
    print(batch_features, batch_labels)

tensor([[-1.9629, -0.9923],
        [-2.8954,  1.9769]]) tensor([0., 0.])
tensor([[-1.1402, -0.8388],
        [ 1.7273, -1.1858]]) tensor([0., 1.])
tensor([[ 1.0683, -0.9701],
        [ 1.8997,  0.8344]]) tensor([1., 1.])
tensor([[-0.7206, -0.9606],
        [-0.5872, -1.9717]]) tensor([0., 0.])
tensor([[ 1.7774,  1.5116],
        [-0.9382, -0.5430]]) tensor([1., 1.])


# Parallelization in data loader

In [69]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=4) #num_workers---multiple workers helps in parallelization

for batch_features, batch_labels in dataloader:
    print(batch_features, batch_labels)

tensor([[-0.7206, -0.9606],
        [ 1.8997,  0.8344]]) tensor([0., 1.])
tensor([[-1.9629, -0.9923],
        [-2.8954,  1.9769]]) tensor([0., 0.])
tensor([[ 1.7273, -1.1858],
        [-1.1402, -0.8388]]) tensor([1., 0.])
tensor([[-0.5872, -1.9717],
        [-0.9382, -0.5430]]) tensor([0., 1.])
tensor([[ 1.0683, -0.9701],
        [ 1.7774,  1.5116]]) tensor([1., 1.])


# Sampler in dataloader
It is responsible for shuffling data in data loader

### It is of 2 Types
- #### Sequential Sampler
    - [1,2,3,4,5,6,7,8,9,10] -> [1,2,3,4,5],[6,7,8,9,10]
    - Generally used in Time series data where we do not want data to get shuffled
    - Default when shuffle=False
- #### Random Sampler
    - [1,2,3,4,5,6,7,8,9,10] -> [9,1,5,3,6],[2,8,4,6,10]
    - Default when shuffle=True
- #### Custom sampler
    - we can define our own conditions for sampler

# Collate Function in dataloader
After \_\_get_item\_\_ function fetches data requested by sampler in data loader there is a need for a method to combine all that data into a single batch

#### Example of a situation of when we might need a method that requires a customized collate method
| Sentence               | Tokenized Data | Label |
|------------------------|----------------|-------|
| "I love coding"        | [1,2,3]        | 0     |
| "Deep Learning Rocks"  | [4,5,6]        | 1     |
| "Transformers are fun" | [7,8,9,10]     | 1     |
| "Hello World"          | [11,12]        | 0     |

It is easier to merge sentence 1 and 2 but since sentence 3 and 4 are of different size we need to write a custom code  by writing 2 zeros after hello worlds tokens

# Attributes in Datloader
pin_memory      -> Training happens in GPU if it's set to 2
num_worker      -> parallelization
drop_last=True  -> in a dataset of 82 rows we have a batch size of 10. Here we have last 2 rows which are not part of any batch. hence they are dropped
collate_fn  // discussed above (pass own fn)
sampler // discussed above (pass own fn)

In [70]:
class CustomDataset1(Dataset):
    def __init__(self, fe, labels):
        self.features = fe
        self.labels = labels

    def __len__(self):
        return self.features.shape[0]
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


In [75]:
batch_size =32
learning_rate =0.1
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [76]:
model4 = PredictionModel(X_train_tensor.shape[1])
optimizer = torch.optim.SGD(model4.parameters(), lr=learning_rate)
loss_function = nn.BCELoss()

In [77]:
epochs = 25
for epoch in range(epochs):
    for batch_f, batch_labels in train_loader:
        y_pred = model4(batch_f.float())
        loss = loss_function(y_pred, batch_labels.reshape(-1,1).float())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


    print("epoch : ",epoch+1 , " loss : ",loss.item())

epoch :  1  loss :  0.5765255689620972
epoch :  2  loss :  0.6043629050254822
epoch :  3  loss :  0.2998431622982025
epoch :  4  loss :  0.2349870651960373
epoch :  5  loss :  0.18276910483837128
epoch :  6  loss :  0.07843217998743057
epoch :  7  loss :  0.04223843291401863
epoch :  8  loss :  0.014157005585730076
epoch :  9  loss :  0.01871477998793125
epoch :  10  loss :  0.1649448573589325
epoch :  11  loss :  0.07795923948287964
epoch :  12  loss :  0.008943992666900158
epoch :  13  loss :  0.0007818022859282792
epoch :  14  loss :  0.00931872520595789
epoch :  15  loss :  0.002011149888858199
epoch :  16  loss :  0.021395474672317505
epoch :  17  loss :  0.0038361570332199335
epoch :  18  loss :  0.016433119773864746
epoch :  19  loss :  0.0026999707333743572
epoch :  20  loss :  0.03595316782593727
epoch :  21  loss :  0.0236111581325531
epoch :  22  loss :  0.8716157078742981
epoch :  23  loss :  0.12657003104686737
epoch :  24  loss :  0.038007814437150955
epoch :  25  loss : 

In [79]:
model4.eval() # set model to evaluation mode
accuracy_list =  []
with torch.no_grad():
    for batch_f, batch_labels in test_loader:
        y_pred = model4(batch_f.float())
        y_pred = (y_pred > 0.7).float()
        batch_accuracy = (y_pred == batch_labels).float().mean().item()
        accuracy_list.append(batch_accuracy)

overall_accuracy = sum(accuracy_list)/len(accuracy_list)
print(f'Overall accuracy : {overall_accuracy}')


Overall accuracy : 0.5322265625
